# <font color="#418FDE" size="6.5" uppercase>**Numerical Methods**</font>

>Last update: 20260609.
    
By the end of this Lecture, you will be able to:
- Implement simple iterative methods for solving engineering equations. 
- Evaluate convergence behavior using tolerances and iteration limits. 
- Package numerical procedures into reusable Python functions. 


## **1. Bisection Root Finding**

### **1.1. Bracket the Root**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Chemical Engineers/Module_03/Lecture_B/image_01_01.jpg?v=1780981543" width="250">



>* Choose bounds with opposite function signs
>* Continuity ensures a reliable root interval

>* Test reasonable values for sign changes
>* Keep brackets tight but root-containing

>* Good brackets make bisection reliable
>* Continuity and sign changes need careful checking



In [ ]:
#@title Python Code - Bracket the Root

# This example brackets an engineering root.
# Opposite signs indicate a crossing.
# The interval then supports bisection.

# Define a simple tank balance function.
def net_flow_error(flow_lpm, target_lpm):
    return flow_lpm - target_lpm

# Store two trial flow rates.
lower_flow = 35.0
upper_flow = 80.0

# Store the desired balance point.
target_flow = 52.0
lower_value = net_flow_error(lower_flow, target_flow)
upper_value = net_flow_error(upper_flow, target_flow)

# Check whether signs are opposite.
bracket_found = lower_value * upper_value < 0.0
status_text = "valid bracket" if bracket_found else "not bracketed"

# Print compact endpoint information.
print("Bisection bracketing check for flow balance")
print(f"lower: {lower_flow:.1f} L/min, f = {lower_value:.1f}")
print(f"upper: {upper_flow:.1f} L/min, f = {upper_value:.1f}")
print(f"Result: {status_text}")

# Report the enclosed interval safely.
if bracket_found:
    print(f"Root lies between {lower_flow:.1f} and {upper_flow:.1f} L/min")

# Show what a failed bracket looks like.
bad_lower = 10.0
bad_upper = 30.0

# Evaluate the second trial interval.
bad_lower_value = net_flow_error(bad_lower, target_flow)
bad_upper_value = net_flow_error(bad_upper, target_flow)

# Test the second sign product.
bad_bracket = bad_lower_value * bad_upper_value < 0.0
bad_status = "valid bracket" if bad_bracket else "not bracketed"

# Print one contrast line.
print(f"Trial [{bad_lower:.0f}, {bad_upper:.0f}] is {bad_status}")



### **1.2. Root Interval**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Chemical Engineers/Module_03/Lecture_B/image_01_02.jpg?v=1780981545" width="250">



>* Root interval brackets a sign-changing solution
>* Repeated halving reduces uncertainty toward precision

>* Test midpoint signs to choose the half
>* Keep bracketing the root while narrowing choices

>* Halving intervals tracks bisection progress.
>* Narrower intervals show greater solution confidence.



In [ ]:
#@title Python Code - Root Interval

# Bisection keeps a trustworthy root interval.
# Each midpoint test halves remaining uncertainty.
# Engineering equations often need reliable brackets.

# Define a simple engineering balance function.
def residual(flow_rate_gpm):
    return flow_rate_gpm**2 - 100.0

# Store endpoints for the initial bracket.
left = 5.0
right = 15.0

# Evaluate endpoint residual signs safely.
f_left = residual(left)
f_right = residual(right)

# Confirm the interval brackets a root.
if f_left * f_right > 0:
    raise ValueError("Endpoints must have opposite signs.")

# Print a compact table heading.
print("iter | left | right | width")

# Update the root interval several times.
for iteration in range(1, 7):
    midpoint = (left + right) / 2.0
    f_mid = residual(midpoint)

    # Keep the half with sign change.
    if f_left * f_mid <= 0:
        right = midpoint
        f_right = f_mid
    else:
        left = midpoint
        f_left = f_mid

    # Show how uncertainty shrinks.
    width = right - left
    print(f"{iteration:4d} | {left:5.2f} | {right:5.2f} | {width:5.2f}")

# Report the final estimated root.
estimate = (left + right) / 2.0
print(f"Estimated flow rate: {estimate:.3f} gpm")



### **1.3. Tolerance Criteria**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Chemical Engineers/Module_03/Lecture_B/image_01_03.jpg?v=1780981547" width="250">



>* Tolerance tells bisection when to stop
>* Choose tolerance to match engineering accuracy needs

>* Check interval width or midpoint function value
>* Choose tolerance based on system behavior

>* Balance accuracy with computational effort
>* Match tolerance and set iteration limits



In [ ]:
#@title Python Code - Tolerance Criteria

# Bisection tolerance controls engineering accuracy.
# We solve a simple pressure equation.
# Iteration limits protect numerical calculations.

# Define the engineering residual function.
def pressure_residual(volume_liters):
    return volume_liters**3 - 10.0

# Create a reusable bisection solver.
def bisection_solver(function, lower, upper, tolerance, max_iterations):
    f_lower = function(lower)
    f_upper = function(upper)

    # Check that the bracket contains a root.
    if f_lower * f_upper > 0:
        raise ValueError("Initial bracket must straddle the root")

    # Iterate until tolerance or limit is reached.
    for iteration in range(1, max_iterations + 1):
        midpoint = (lower + upper) / 2.0
        f_midpoint = function(midpoint)

        # Estimate uncertainty using half interval width.
        half_width = (upper - lower) / 2.0
        if half_width <= tolerance:
            return midpoint, iteration, half_width, f_midpoint

        # Keep the subinterval containing the root.
        if f_lower * f_midpoint <= 0:
            upper = midpoint
            f_upper = f_midpoint

        else:
            lower = midpoint
            f_lower = f_midpoint

    # Return best estimate after iteration limit.
    midpoint = (lower + upper) / 2.0
    half_width = (upper - lower) / 2.0
    return midpoint, max_iterations, half_width, function(midpoint)

# Compare loose and tight tolerance choices.
loose_result = bisection_solver(
    pressure_residual, 1.0, 3.0, 0.05, 50
)

tight_result = bisection_solver(
    pressure_residual, 1.0, 3.0, 0.0001, 50
)

# Print compact engineering interpretation.
print("Bisection tolerance comparison for V^3 - 10 = 0")
print("Loose tolerance root: {:.5f} L".format(loose_result[0]))
print("Loose iterations used: {}".format(loose_result[1]))
print("Loose uncertainty: ±{:.5f} L".format(loose_result[2]))
print("Tight tolerance root: {:.5f} L".format(tight_result[0]))
print("Tight iterations used: {}".format(tight_result[1]))
print("Tight uncertainty: ±{:.5f} L".format(tight_result[2]))
print("Tighter tolerance improves accuracy but costs iterations.")



## **2. Iteration Control**

### **2.1. Tolerance Criteria**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Chemical Engineers/Module_03/Lecture_B/image_02_01.jpg?v=1780981549" width="250">



>* Tolerance defines enough accuracy for the task
>* Stop iterations to balance accuracy and effort

>* Match tolerance to engineering context
>* Avoid overly loose or strict limits

>* Use tolerances to track convergence progress
>* Check multiple signs before trusting results



In [ ]:
#@title Python Code - Tolerance Criteria

# Tolerances decide when iterations should stop.
# Residuals measure remaining engineering equation mismatch.
# Iteration limits prevent endless numerical work.

import math

# Define a small reusable bisection solver.
def bisection_solver(function, lower, upper, tolerance, maximum_iterations):
    residual_lower = function(lower)

    residual_upper = function(upper)
    if residual_lower * residual_upper > 0:
        raise ValueError("Interval must bracket one root")

    records = []
    for iteration in range(1, maximum_iterations + 1):
        midpoint = (lower + upper) / 2.0

        residual_midpoint = function(midpoint)
        interval_width = abs(upper - lower) / 2.0
        records.append((iteration, midpoint, residual_midpoint, interval_width))

        if abs(residual_midpoint) <= tolerance:
            return midpoint, iteration, "residual tolerance", records

        if interval_width <= tolerance:
            return midpoint, iteration, "change tolerance", records

        if residual_lower * residual_midpoint < 0:
            upper = midpoint
            residual_upper = residual_midpoint

        else:
            lower = midpoint
            residual_lower = residual_midpoint

    return midpoint, maximum_iterations, "iteration limit", records

# Model friction factor using the Colebrook equation.
def colebrook_residual(friction_factor):
    roughness = 0.00015

    diameter = 0.25
    reynolds = 100000.0
    left_term = 1.0 / math.sqrt(friction_factor)

    right_argument = roughness / (3.7 * diameter)
    right_argument += 2.51 / (reynolds * math.sqrt(friction_factor))
    return left_term + 2.0 * math.log10(right_argument)

# Compare loose and strict tolerance settings.
loose_result = bisection_solver(colebrook_residual, 0.008, 0.08, 1e-3, 40)
strict_result = bisection_solver(colebrook_residual, 0.008, 0.08, 1e-8, 40)
limited_result = bisection_solver(colebrook_residual, 0.008, 0.08, 1e-12, 5)

# Summarize outcomes without printing long iteration logs.
print("Colebrook friction factor by bisection")
print(f"Loose tolerance: f={loose_result[0]:.6f}, iterations={loose_result[1]}, stop={loose_result[2]}")
print(f"Strict tolerance: f={strict_result[0]:.6f}, iterations={strict_result[1]}, stop={strict_result[2]}")
print(f"Iteration limit: f={limited_result[0]:.6f}, iterations={limited_result[1]}, stop={limited_result[2]}")

# Show final residuals for engineering meaning.
loose_residual = abs(colebrook_residual(loose_result[0]))
strict_residual = abs(colebrook_residual(strict_result[0]))
limited_residual = abs(colebrook_residual(limited_result[0]))

print(f"Loose residual magnitude: {loose_residual:.2e}")
print(f"Strict residual magnitude: {strict_residual:.2e}")
print(f"Limited residual magnitude: {limited_residual:.2e}")



### **2.2. Tolerance Criteria**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Chemical Engineers/Module_03/Lecture_B/image_02_02.jpg?v=1780981553" width="250">



>* Tolerance sets when iterations are good enough
>* Avoids unnecessary precision and computation time

>* Match tolerance to engineering context
>* Balance accuracy with practical computational cost

>* Use tolerances to judge convergence speed
>* Report convergence status with final results



In [ ]:
#@title Python Code - Tolerance Criteria

# Iteration control needs clear stopping rules.
# Tolerances compare estimates during numerical solving.
# Limits prevent endless engineering calculations.

# Define one reusable fixed point solver.
def fixed_point_solver(start_value, tolerance, maximum_steps):
    estimate = float(start_value)

    history = []
    converged = False

    for step in range(1, maximum_steps + 1):
        new_estimate = 0.5 * (estimate + 25.0 / estimate)

        absolute_change = abs(new_estimate - estimate)
        relative_change = absolute_change / max(abs(new_estimate), 1e-12)

        history.append((step, new_estimate, absolute_change, relative_change))
        estimate = new_estimate

        if absolute_change < tolerance:
            converged = True
            break

    return estimate, converged, history

# Run a relaxed tolerance case.
relaxed_result = fixed_point_solver(2.0, 1e-3, 20)
relaxed_value, relaxed_done, relaxed_history = relaxed_result

# Run a strict tolerance case.
strict_result = fixed_point_solver(2.0, 1e-10, 4)
strict_value, strict_done, strict_history = strict_result

# Display concise convergence summaries.
print("Solving x^2 = 25 by fixed point iteration")
print(f"Relaxed tolerance converged: {relaxed_done}")
print(f"Relaxed final estimate: {relaxed_value:.6f}")
print(f"Relaxed iterations used: {len(relaxed_history)}")

print(f"Strict tolerance converged: {strict_done}")
print(f"Strict final estimate: {strict_value:.6f}")
print(f"Strict iterations used: {len(strict_history)}")

# Inspect the final strict change.
last_step = strict_history[-1]
print(f"Last strict absolute change: {last_step[2]:.3e}")



### **2.3. Convergence Criteria**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Chemical Engineers/Module_03/Lecture_B/image_02_03.jpg?v=1780981551" width="250">



>* Judge when approximations are good enough
>* Stop when changes become insignificant

>* Match criteria to math and problem purpose
>* Balance accuracy needs against wasted iterations

>* Iteration limits prevent endless failed calculations
>* Unmet convergence signals methods need revision



In [ ]:
#@title Python Code - Convergence Criteria

# Iteration control prevents endless numerical calculations.
# Convergence tolerances define acceptable engineering accuracy.
# Iteration limits flag unsuccessful numerical searches.

# Define a reusable bisection solver.
def bisection_solve(function, low, high, tolerance, maximum_iterations):
    lower_value = function(low)
    upper_value = function(high)

    if lower_value * upper_value > 0:
        raise ValueError("Interval must bracket one root")

    history = []
    midpoint = (low + high) / 2

    for iteration in range(1, maximum_iterations + 1):
        midpoint = (low + high) / 2
        mid_value = function(midpoint)

        interval_width = abs(high - low)
        history.append((iteration, midpoint, interval_width, mid_value))

        if interval_width < tolerance or abs(mid_value) < tolerance:
            return midpoint, iteration, True, history

        if lower_value * mid_value < 0:
            high = midpoint
            upper_value = mid_value

        else:
            low = midpoint
            lower_value = mid_value

    return midpoint, maximum_iterations, False, history

# Model a pipe diameter equation.
def pressure_balance(diameter_meters):
    flow_term = 0.04 / (diameter_meters ** 4)
    return flow_term - 2500

# Solve with a reasonable tolerance.
root, steps, converged, history = bisection_solve(
    pressure_balance, 0.05, 0.20, 1e-5, 40
)

# Solve with an unrealistically small iteration limit.
rough_root, rough_steps, rough_converged, rough_history = bisection_solve(
    pressure_balance, 0.05, 0.20, 1e-9, 5
)

# Report whether each run satisfied convergence.
print(f"Good run converged: {converged}, iterations: {steps}")
print(f"Estimated diameter: {root:.5f} meters")
print(f"Limited run converged: {rough_converged}, iterations: {rough_steps}")
print(f"Limited estimate: {rough_root:.5f} meters")
print("Last interval widths from good run:")

# Show only the final few convergence indicators.
for item in history[-3:]:
    iteration, estimate, width, residual = item
    print(f"iter {iteration}: width={width:.2e}, residual={residual:.2e}")



## **3. Reusable Solver Functions**

### **3.1. Solver Structure**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Chemical Engineers/Module_03/Lecture_B/image_03_01.jpg?v=1780981538" width="250">



>* Clear inputs guide repeated solver steps
>* Simple interfaces make solvers reusable

>* Initialize values, then repeatedly update estimates
>* Check convergence before returning readable results

>* Return convergence details, not just answers
>* Report failures so users can adjust



In [ ]:
#@title Python Code - Solver Structure

# Reusable solvers hide repeated numerical steps.
# This example solves a pressure equation.
# The structure returns value and status.

# Define a reusable fixed point solver.
def fixed_point_solver(update_function, initial_guess, tolerance, max_iterations):
    current_guess = float(initial_guess)
    last_change = float("inf")

    # Repeat until convergence or iteration limit.
    for iteration_number in range(1, max_iterations + 1):
        next_guess = update_function(current_guess)
        last_change = abs(next_guess - current_guess)

        # Stop when the update is sufficiently small.
        if last_change <= tolerance:
            return next_guess, iteration_number, True, last_change

        current_guess = next_guess

    # Return clearly when convergence was not reached.
    return current_guess, max_iterations, False, last_change

# Pipe pressure drop is represented by this update.
def pressure_update(previous_drop):
    base_drop = 12.0
    coupling_factor = 0.25
    return base_drop + coupling_factor * previous_drop

# Choose solver settings for the engineering estimate.
initial_pressure_drop = 0.0
tolerance_value = 0.001
maximum_iterations = 25

# Run the solver through its simple interface.
result = fixed_point_solver(
    pressure_update,
    initial_pressure_drop,
    tolerance_value,
    maximum_iterations,
)

# Unpack the returned engineering information.
pressure_drop, iterations, converged, final_change = result
status_text = "converged" if converged else "stopped early"

# Print a compact summary for engineering judgment.
print("Reusable fixed-point solver demonstration")
print(f"Status: {status_text}")
print(f"Pressure drop estimate: {pressure_drop:.3f} psi")
print(f"Iterations used: {iterations}")
print(f"Final change: {final_change:.6f} psi")



### **3.2. Reusable Solver Design**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Chemical Engineers/Module_03/Lecture_B/image_03_02.jpg?v=1780981540" width="250">



>* Write methods once for many problems
>* Separate model, method, and result interpretation

>* Use clear inputs and outputs
>* Avoid hard-coded problem details

>* Return convergence details, not just estimates
>* Make solvers transparent, reliable, and reusable



In [ ]:
#@title Python Code - Reusable Solver Design

# Reusable solvers separate methods from models.
# Clear inputs make engineering calculations adaptable.
# Returned diagnostics help judge convergence quality.

# No extra package installation is required.

# Define reusable fixed-point iteration solver.
def fixed_point_solver(update_function, initial_guess, tolerance, max_iterations):

    # Store current estimate and starting error.
    estimate = float(initial_guess)
    error = float("inf")

    # Iterate until tolerance or limit stops.
    for iteration in range(1, max_iterations + 1):

        # Compute next estimate from supplied model.
        new_estimate = float(update_function(estimate))
        error = abs(new_estimate - estimate)

        # Accept estimate when change becomes small.
        if error <= tolerance:
            return new_estimate, True, iteration, error

        # Continue with the newest estimate.
        estimate = new_estimate

    # Report nonconvergence with final available estimate.
    return estimate, False, max_iterations, error

# Model pressure drop using a simple pipe relation.
def pipe_flow_update(flow_gpm):

    # Target pressure drop is eight psi.
    target_drop = 8.0
    coefficient = 0.015

    # Rearranged equation gives updated flow estimate.
    updated_flow = (target_drop / coefficient) ** 0.5
    return updated_flow

# Model tank depth using metric volume relation.
def tank_depth_update(depth_m):

    # Cylindrical tank area uses square meters.
    volume_m3 = 12.0
    area_m2 = 3.0

    # Rearranged equation gives updated depth estimate.
    updated_depth = volume_m3 / area_m2
    return updated_depth

# Solve two different engineering problems.
pipe_result = fixed_point_solver(pipe_flow_update, 10.0, 0.001, 25)
tank_result = fixed_point_solver(tank_depth_update, 1.0, 0.001, 25)

# Unpack results for readable output.
pipe_value, pipe_ok, pipe_steps, pipe_error = pipe_result
tank_value, tank_ok, tank_steps, tank_error = tank_result

# Print compact convergence diagnostics.
print("Reusable fixed-point solver demonstration")
print(f"Pipe flow: {pipe_value:.2f} gpm, converged={pipe_ok}")
print(f"Pipe iterations={pipe_steps}, final change={pipe_error:.4f}")
print(f"Tank depth: {tank_value:.2f} m, converged={tank_ok}")
print(f"Tank iterations={tank_steps}, final change={tank_error:.4f}")



### **3.3. Error Handling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Chemical Engineers/Module_03/Lecture_B/image_03_03.jpg?v=1780981542" width="250">



>* Check inputs and convergence before trusting results
>* Build safer, reusable solver functions

>* Stop early for invalid solver inputs
>* Report nonconvergence with best available estimate

>* Predictable failures support automated solver workflows
>* Diagnostics guide responsible engineering decisions



In [ ]:
#@title Python Code - Error Handling

# Build reusable solvers with clear failures.
# Validate engineering inputs before iterating.
# Return status information with estimates.

def heat_balance_residual(temp_c, flow_kg_s, cp_kj_kg_c, heat_kw):
    return flow_kg_s * cp_kj_kg_c * temp_c - heat_kw

# Define a reusable bisection solver.
def bisection_solver(function, lower, upper, tolerance, max_iterations, args=()):
    if tolerance <= 0 or max_iterations < 1:
        raise ValueError("tolerance must be positive and iterations positive")

    if not isinstance(max_iterations, int):
        raise TypeError("max_iterations must be an integer value")

    f_lower = function(lower, *args)
    f_upper = function(upper, *args)

    if f_lower * f_upper > 0:
        raise ValueError("bounds must bracket a sign change")

    for iteration in range(1, max_iterations + 1):
        midpoint = (lower + upper) / 2
        f_midpoint = function(midpoint, *args)
        error_estimate = abs(upper - lower) / 2

        if abs(f_midpoint) <= tolerance or error_estimate <= tolerance:
            return midpoint, iteration, True

        if f_lower * f_midpoint < 0:
            upper, f_upper = midpoint, f_midpoint
        else:
            lower, f_lower = midpoint, f_midpoint

    return midpoint, max_iterations, False

# Store one physically meaningful case.
case = (2.0, 4.18, 250.0)

# Solve for exchanger temperature rise.
solution = bisection_solver(
    heat_balance_residual, 0.0, 100.0, 1e-6, 30, case
)

# Unpack the solver diagnostic information.
temperature, iterations, converged = solution

print(f"Temperature estimate: {temperature:.2f} deg C")
print(f"Iterations used: {iterations}")
print(f"Converged: {converged}")

# Demonstrate clean handling of invalid brackets.
try:
    bad_solution = bisection_solver(
        heat_balance_residual, 40.0, 60.0, 1e-6, 30, case
    )
except ValueError as error_message:
    print(f"Handled error: {error_message}")

# Demonstrate nonconvergence status without crashing.
limited_solution = bisection_solver(
    heat_balance_residual, 0.0, 100.0, 1e-12, 3, case
)

print(f"Limited run converged: {limited_solution[2]}")



# <font color="#418FDE" size="6.5" uppercase>**Numerical Methods**</font>


In this lecture, you learned to:
- Implement simple iterative methods for solving engineering equations. 
- Evaluate convergence behavior using tolerances and iteration limits. 
- Package numerical procedures into reusable Python functions. 

In the next Module (Module 4), we will go over 'None'